# Chapter 15 &mdash; Mapping Reductions: $A \le_m B$

**Concept 7 of the Chapter 15 decomposition:** *Mapping Reductions: $A\leq_m B$*

A computable $f$ with $x\in A \iff f(x)\in B$; a decider for $B$ then decides $A$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-Mapping-Reductions/Concept-Mapping-Reductions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$A \le_m B \quad\text{iff}\quad \exists \text{ computable } f:\ x\in A \iff f(x)\in B$$

$f$ is a **total computable function on strings**. It does **not** run anything; it
**transforms** one question into another.

The two consequences you will use constantly:

* if $B$ is **decidable** and $A \le_m B$, then $A$ is decidable;
* **contrapositive:** if $A$ is **undecidable** and $A \le_m B$, then $B$ is
  undecidable.

Two cautions. Mapping reducibility is **not symmetric**: $A\le_m B$ does not give
$B\le_m A$. And $A \le_m B$ implies $\overline{A} \le_m \overline{B}$ &mdash; which is
why you cannot map-reduce $A_{TM}$ to $\overline{A_{TM}}$.

## 2. Definitions

### A mapping reduction between two decidable problems

In [ ]:
# A = "this binary numeral is even"        B = "this string ends in 0"
def in_A(x): return x != '' and int(x, 2) % 2 == 0
def in_B(x): return x.endswith('0')
def f(x):    return x                       # the identity works here

# A2 = "even number of 0s"  ->  B2 = "even number of 1s", via a real
# transform: swap the two symbols.  (Dropping a bit to halve a numeral is
# the tempting choice and it is WRONG: n div 2 being even means n mod 4 is
# 0 or 1, not n mod 4 = 0.)
def in_A2(x): return x.count('0') % 2 == 0
def in_B2(x): return x.count('1') % 2 == 0
def f2(x):    return x.translate(str.maketrans('01', '10'))

### Checking a reduction mechanically

In [ ]:
from itertools import product
def check_reduction(in_A, f, in_B, upto=10, minlen=1):
    xs = [''.join(p) for k in range(minlen, upto + 1)
          for p in product('01', repeat=k)]
    bad = [x for x in xs if in_A(x) != in_B(f(x))]
    return (not bad), bad[:5], len(xs)

## 3. Tests

The identity is a legitimate reduction when the questions coincide.

In [ ]:
ok, bad, n = check_reduction(in_A, f, in_B)
print("A <=m B via identity? %s  (checked %d strings)" % (ok, n))
assert ok

A reduction that actually **transforms** the instance.

In [ ]:
ok, bad, n = check_reduction(in_A2, f2, in_B2)
print("A2 <=m B2 via 'swap 0 and 1'? %s  (checked %d)" % (ok, n))
print("counterexamples :", bad)
assert ok
for x in ['100', '1000', '110', '10']:
    print("   %-6r in A2? %-6s f2(x)=%-6r in B2? %s"
          % (x, in_A2(x), f2(x), in_B2(f2(x))))

**Decidability transfers forwards.**

In [ ]:
def decider_for_B2(x): return in_B2(x)
def derived_decider_for_A2(x): return decider_for_B2(f2(x))
from itertools import product
xs = [''.join(p) for k in range(1, 9) for p in product('01', repeat=k)]
assert all(derived_decider_for_A2(x) == in_A2(x) for x in xs)
print("a decider for B2, composed with f, decides A2 -- on all %d strings" % len(xs))

**Undecidability transfers backwards** &mdash; the contrapositive, which is the useful half.

In [ ]:
print("A <=m B  and  B decidable   =>  A decidable")
print("A <=m B  and  A UNdecidable =>  B UNdecidable")
print()
print("So to show B is hard: find a known-hard A and a computable f.")

**Not symmetric.** A reduction one way says nothing about the other.

In [ ]:
def in_TRIV(x): return True                 # a trivially decidable language
def g(x): return x
ok, _, _ = check_reduction(in_A2, lambda x: '0' if in_A2(x) else '1',
                           lambda y: y == '0')
print("A2 <=m {'0'} ? ", ok)
print()
print("Every decidable language reduces to almost any other, but a hard")
print("language does NOT reduce back.  Direction carries the information.")

And complements come along for the ride.

In [ ]:
ok, _, _ = check_reduction(lambda x: not in_A2(x), f2, lambda y: not in_B2(y))
print("complement(A2) <=m complement(B2) via the same f? ", ok)
assert ok
print("\nThat is why A_TM cannot be map-reduced to its own complement:")
print("it would force complement(A_TM) to be RE, and it is not.")

## 4. Exercises


1. Show $\le_m$ is transitive.
2. Why must $f$ be **total**? What breaks if it can diverge?
3. Give two languages with $A\le_m B$ but not $B\le_m A$.

In [ ]:
# Your work for the exercises above.